# Fraud Investigation Agent
## 1. Dataset Exploration — PaySim

Цель этапа: изучить структуру данных, целевую переменную,
распределения признаков и качество исходного датасета.

In [4]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 3.0.6
numpy: 2.4.6


In [9]:
from pathlib import Path

data_path = Path(r"D:\Fraud_project\fraud-investigation-agent\data\paysim.csv")

In [10]:
df = pd.read_csv(data_path)

print("Данные загружены!")
print("Размер таблицы:", df.shape)

Данные загружены!
Размер таблицы: (6362620, 11)


In [11]:
display(df.head())

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB


## 2. Первичная проверка данных

На этом этапе изучаем распределение целевой переменной, проверяем пропуски и дубликаты, а также знакомимся с типами операций. Это поможет определить особенности датасета до подготовки признаков и обучения модели.

In [13]:
print("Распределение isFraud:")
print(df["isFraud"].value_counts())

Распределение isFraud:
isFraud
0    6354407
1       8213
Name: count, dtype: int64


In [17]:
print("Доли классов, %:")
print((df["isFraud"].value_counts(normalize=True)).round(4))

Доли классов, %:
isFraud
0    0.9987
1    0.0013
Name: proportion, dtype: float64


In [20]:
print("Количество пропусков по столбцам:")
display(df.isna().sum().to_frame("missing_count"))

Количество пропусков по столбцам:


,missing_count
step,0
type,0
amount,0
nameOrig,0
oldbalanceOrg,0
newbalanceOrig,0
nameDest,0
oldbalanceDest,0
newbalanceDest,0
isFraud,0


In [21]:
print("Полных дубликатов:", df.duplicated().sum())

Полных дубликатов: 0


In [22]:
print("Количество операций каждого типа:")
display(df["type"].value_counts().to_frame("count"))

Количество операций каждого типа:


,count
type,
CASH_OUT,2237500
PAYMENT,2151495
CASH_IN,1399284
TRANSFER,532909
DEBIT,41432


In [27]:
print("Мошеннические операции по типам:")
display(
    df.groupby("type")["isFraud"]
    .agg(["count", "sum", "mean"])
    .rename(columns={
        "count": "total_operations",
        "sum": "fraud_operations",
        "mean": "fraud_rate"
    })
    .sort_values("fraud_operations", ascending=False)
)

Мошеннические операции по типам:


,total_operations,fraud_operations,fraud_rate
type,,,
CASH_OUT,2237500,4116,0.001840
TRANSFER,532909,4097,0.007688
CASH_IN,1399284,0,0.000000
DEBIT,41432,0,0.000000
PAYMENT,2151495,0,0.000000


## 3. Анализ признаков и качества разметки

Изучаем суммы транзакций, изменения балансов и связь целевой переменной `isFraud` с флагом `isFlaggedFraud`. Проверяем, какие признаки могут быть полезны для обнаружения мошенничества.

In [28]:
print("Распределение isFlaggedFraud:")
print(df["isFlaggedFraud"].value_counts())

Распределение isFlaggedFraud:
isFlaggedFraud
0    6362604
1         16
Name: count, dtype: int64


In [30]:
print("\nСравнение isFraud и isFlaggedFraud:")
display(
    pd.crosstab(
        df["isFraud"],
        df["isFlaggedFraud"],
        rownames=['Фактическое количество'],
        colnames=['Флаг существующей системы']
    )
)


Сравнение isFraud и isFlaggedFraud:


Флаг существующей системы,0,1
Фактическое количество,,
0,6354407,0
1,8197,16


In [31]:
display(
    df.groupby("isFraud")["amount"]
    .describe()
    .round(2)
)

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,6354407.0,178197.04,596236.98,0.01,13368.40,74684.72,208364.76,92445516.64
1,8213.0,1467967.30,2404252.95,0.00,127091.33,441423.44,1517771.48,10000000.00


In [36]:
df["orig_balance_change"] = (
    df["oldbalanceOrg"] - df["newbalanceOrig"]
) # насколько уменьшился баланс отправителя

df["dest_balance_change"] = (
    df["newbalanceDest"] - df["oldbalanceDest"]
) # насколько увеличился баланс получателя

display(
    df.groupby("isFraud")[
       ["amount",
        "orig_balance_change",
        "dest_balance_change"
       ]    
    ]
    .median()
    .round(2)
)

,amount,orig_balance_change,dest_balance_change
isFraud,,,
0,74684.72,0.00,0.0
1,441423.44,436317.49,0.0


In [37]:
print("Доля операций без изменения баланса отправителя:")
print(
    (df["oldbalanceOrg"] == df["newbalanceOrig"])
    .mean()
    .round(4)
)

print("\nДоля операций без изменения баланса получателя:")
print(
    (df["oldbalanceDest"] == df["newbalanceDest"])
    .mean()
    .round(4)
)

print("\nСредние суммы по классам:")
display(
    df.groupby("isFraud")["amount"]
      .mean()
      .round(2)
)

Доля операций без изменения баланса отправителя:
0.3283

Доля операций без изменения баланса получателя:
0.3642

Средние суммы по классам:


isFraud
0     178197.04
1    1467967.30
Name: amount, dtype: float64

## 4. Сравнение признаков по классам

Сравниваем распределения сумм транзакций и изменений балансов для обычных и мошеннических операций. Проверяем, какие признаки потенциально информативны, и ищем закономерности, которые стоит изучить перед обучением модели.

In [38]:
display(
    df.groupby("isFraud")["amount"]
    .quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.95])
    .unstack()
    .round(2)
)

,0.10,0.25,0.50,0.75,0.90,0.95
isFraud,,,,,,
0,4496.36,13368.40,74684.72,208364.76,364373.44,515610.42
1,37720.99,127091.33,441423.44,1517771.48,4521723.51,8006429.04


In [39]:
display(
    df.groupby("isFraud")["orig_balance_change"]
      .describe(percentiles=[0.25, 0.5, 0.75, 0.9])
      .round(2)
)

,count,mean,std,min,25%,50%,75%,90%,max
isFraud,,,,,,,,,
0,6354407.0,-23141.52,106223.3,-1915267.9,0.00,0.00,10102.13,33713.26,4164236.31
1,8213.0,1457274.97,2396099.2,0.0,124582.58,436317.49,1503034.86,4475400.17,10000000.00


In [40]:
display(
    df[df["isFraud"] == 1]
    .groupby('type')
    .agg(
        fraud_count=("isFraud", "size"),
        median_amount=("amount", "median"),
        mean_amount=("amount", "mean")
    )
    .round(2)
    .sort_values("fraud_count", ascending=False)
)

,fraud_count,median_amount,mean_amount
type,,,
CASH_OUT,4116,435516.90,1455102.59
TRANSFER,4097,445705.76,1480891.67


In [41]:
# Насколько уменьшение баланса отправителя соответствует сумме операции
df["orig_balance_diff"] = (
    df["oldbalanceOrg"] - df["newbalanceOrig"] - df["amount"]
)

# Насколько увеличение баланса получателя соответствует сумме операции
df["dest_balance_diff"] = (
    df["newbalanceDest"] - df["oldbalanceDest"] - df["amount"]
)

display(
    df.groupby("isFraud")[
        ["orig_balance_diff", "dest_balance_diff"]
    ]
    .median()
    .round(2)
)

,orig_balance_diff,dest_balance_diff
isFraud,,
0,-69049.31,-3500.68
1,0.00,-2231.46


In [42]:
display(
    df.groupby(["type", "isFraud"])[
        ["orig_balance_diff", "dest_balance_diff"]
    ]
    .median()
    .round(2)
)

orig_balance_diff  dest_balance_diff
type     isFraud                                      
CASH_IN  0               -286848.82         -248899.81
CASH_OUT 0               -119824.22               0.00
         1                     0.00               0.00
DEBIT    0                    -0.00               0.00
PAYMENT  0                  -380.78           -9482.19
TRANSFER 0               -454033.08               0.00
         1                     0.00         -442722.01

In [43]:
display(
    df.groupby(["type", "isFraud"])
      .agg(
          operations=("amount", "size"),
          orig_diff_zero=(
              "orig_balance_diff",
              lambda s: (s == 0).mean()
          ),
          dest_diff_zero=(
              "dest_balance_diff",
              lambda s: (s == 0).mean()
          )
      )
      .round(4)
)

operations  orig_diff_zero  dest_diff_zero
type     isFraud                                            
CASH_IN  0           1399284          0.0000          0.0000
CASH_OUT 0           2233384          0.0602          0.2701
         1              4116          0.9939          0.5275
DEBIT    0             41432          0.1472          0.0400
PAYMENT  0           2151495          0.1304          0.0000
TRANSFER 0            528812          0.0211          0.2900
         1              4097          0.9893          0.0000

In [44]:
print("Столбцы исходного датасета:")
print([
    "step", "type", "amount",
    "nameOrig", "oldbalanceOrg", "newbalanceOrig",
    "nameDest", "oldbalanceDest", "newbalanceDest",
    "isFraud", "isFlaggedFraud"
])

Столбцы исходного датасета:
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


## 5. Промежуточные выводы

### Структура данных
- 6 362 620 транзакций и 11 исходных столбцов.
- Пропуски и полные дубликаты не обнаружены.
- Целевая переменная `isFraud` сильно несбалансирована: 8 213 мошеннических операций (около 0,13%).

### Основные наблюдения
- В исследуемом датасете мошеннические операции встречаются только в категориях `CASH_OUT` и `TRANSFER`.
- Медианная и средняя суммы мошеннических операций выше, чем у обычных.
- Для большинства мошеннических операций уменьшение баланса отправителя совпадает с суммой транзакции.
- `isFlaggedFraud` отмечает лишь 16 из 8 213 мошеннических операций, поэтому не рассматривается как входной признак первой модели.

### Ограничения
- PaySim является синтетическим датасетом, поэтому найденные закономерности могут отражать особенности симуляции.
- Наблюдаемые различия еще не подтверждают качество будущей модели.
- Необходимо проверить риск утечки целевой переменной, выбрать схему разделения данных и оценить модель с учетом дисбаланса классов.